In [1]:
# Install required libraries
import sys
!{sys.executable} -m pip install kagglehub pandas


  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)

   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import os
import json
import kagglehub

# Create output folder
output_folder = "medical_prescription_dataset"
os.makedirs(output_folder, exist_ok=True)

# Download dataset
path = kagglehub.dataset_download("nadaarfaoui/ocr-processed-handwritten-prescriptions")

print("Dataset downloaded at:", path)

c:\Users\santo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 15.8M/15.8M [00:05<00:00, 3.22MB/s]

Extracting files...


Dataset downloaded at: C:\Users\santo\.cache\kagglehub\datasets\nadaarfaoui\ocr-processed-handwritten-prescriptions\versions\4


In [3]:
# Check files inside dataset folder
import os

files = os.listdir(path)
print(files)

['data', '__huggingface_repos__.json']


In [4]:
import os
import pandas as pd

data_folder = os.path.join(path, "data")

print(os.listdir(data_folder))

['processed']


In [5]:
import os
import pandas as pd

csv_file = None

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_file = os.path.join(root, file)
            break

print("CSV file found:", csv_file)

df = pd.read_csv(csv_file)

df.head()

CSV file found: C:\Users\santo\.cache\kagglehub\datasets\nadaarfaoui\ocr-processed-handwritten-prescriptions\versions\4\data\processed\test_labels.csv


,IMAGE,MEDICINE_NAME
0,0.png,0
1,1.png,0
2,2.png,0
3,3.png,0
4,4.png,0


In [6]:
df_1000 = df.head(1000)

df_1000.to_csv("prescription_1000.csv", index=False)
df_1000.to_json("prescription_1000.json", orient="records", indent=4)

print("Saved successfully")

Saved successfully


In [7]:
# Save as CSV
csv_output = os.path.join(output_folder, "prescription_1000.csv")
df_1000.to_csv(csv_output, index=False)

print("CSV saved at:", csv_output)

CSV saved at: medical_prescription_dataset\prescription_1000.csv


In [1]:
import json
import os
import uuid

print("Loading existing AI Training Dataset...")

# 1. Load your newly created master dataset
dataset_file = "AI_Training_Dataset.json"
with open(dataset_file, "r", encoding="utf-8") as f:
    training_data = json.load(f)

# 2. Point to your downloaded Kaggle prescription folder
ocr_folder = "medical_prescription_dataset"

print("Scanning for medical prescription images...")
ocr_count = 0

# 3. Loop through the images and format them for VQA
if os.path.exists(ocr_folder):
    for file_name in os.listdir(ocr_folder):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(ocr_folder, file_name)
            
            # Format exactly like our other data
            training_data.append({
                "id": f"ocr_{uuid.uuid4().hex[:8]}",
                "image": image_path,
                "conversations": [
                    {"from": "human", "value": "<image>\nPlease extract the text from this handwritten medical prescription."},
                    # IMPORTANT: If your Kaggle dataset came with a CSV of real transcriptions, 
                    # you would replace this generic string with the real text from the CSV!
                    {"from": "gpt", "value": "[Transcription of prescription would go here]"} 
                ]
            })
            ocr_count += 1
            
            # Let's cap it at 1000 so the dataset stays balanced (you can change this!)
            if ocr_count >= 1000:
                break

# 4. Overwrite the file with the newly added OCR data
with open(dataset_file, "w", encoding="utf-8") as f:
    json.dump(training_data, f, indent=4)

print(f"🎉 Success! Appended {ocr_count} OCR examples.")
print(f"Total training examples is now: {len(training_data)}")

Loading existing AI Training Dataset...
Scanning for medical prescription images...
🎉 Success! Appended 0 OCR examples.
Total training examples is now: 138930


In [3]:
import json
import pandas as pd
import os

print("Updating AI Dataset with actual Prescription Labels...")

# 1. Load the existing dataset
dataset_file = "AI_Training_Dataset.json"
with open(dataset_file, "r", encoding="utf-8") as f:
    training_data = json.load(f)

# 2. Filter out the old dummy OCR data we just made
training_data = [item for item in training_data if not item["id"].startswith("ocr_")]

# 3. Load your prescription CSV
csv_path = os.path.join("medical_prescription_dataset", "prescription_1000.csv")
df_ocr = pd.read_csv(csv_path)

# 4. Map the real labels!
added_count = 0
for index, row in df_ocr.iterrows():
    img_name = str(row['IMAGE'])
    medicine_label = str(row['MEDICINE_NAME'])
    img_path = os.path.join("medical_prescription_dataset", img_name)
    
    # Only add if the image actually exists in the folder
    if os.path.exists(img_path):
        training_data.append({
            "id": f"ocr_{img_name.split('.')[0]}",
            "image": img_path,
            "conversations": [
                {"from": "human", "value": "<image>\nWhat medicine is prescribed in this image?"},
                {"from": "gpt", "value": f"The prescribed medicine class/name is: {medicine_label}"}
            ]
        })
        added_count += 1

# 5. Save the fixed dataset
with open(dataset_file, "w", encoding="utf-8") as f:
    json.dump(training_data, f, indent=4)

print(f"🎉 Success! Replaced dummy data with {added_count} real prescription records.")
print(f"Total training examples ready for the AI: {len(training_data)}")

Updating AI Dataset with actual Prescription Labels...
🎉 Success! Replaced dummy data with 0 real prescription records.
Total training examples ready for the AI: 138930
